In [1]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import sacrebleu
from bert_score import score as bertscore_score
from datasets import Dataset
from peft import PeftModel
from rouge_score import rouge_scorer
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

In [2]:
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
ADAPTER_PATH = Path("agents/models/trained/model_qwen_solver_fast/lora_adapter")
DATA_PATH = "master_incident_dataset.parquet"
OUT_DIR = Path("agents/models/trained/model_qwen_solver/eval")
MAX_SEQ_LEN = 1024
TEST_SIZE = 0.2
SPLIT_SEED = 42
N_EVAL = None  # set to an int like 200 for faster testing, or keep None for full test set

SYSTEM_PROMPT = (
    "You are a Kubernetes Site Reliability Engineering (SRE) agent. "
    "Given raw observability evidence from a Kubernetes incident, provide:\n"
    "1. A root cause diagnosis explaining what went wrong and why.\n"
    "2. A step-by-step fix plan to resolve the incident.\n"
    "3. Concrete actions or commands to apply the fix.\n"
    "4. Verification steps to confirm the fix worked.\n"
    "5. Rollback guidance if the fix causes issues."
)

TARGET_COLS = [
    "diagnosis_text",
    "fix_plan_text",
    "actions_text",
    "verification_text",
    "rollback_text",
]

SECTION_NAMES = ["diagnosis", "fix plan", "actions", "verification", "rollback"]

SCENARIO_HINTS = {
    "createcontainerconfigerror_missing_secret": ["secret", "not found", "createcontainerconfigerror"],
    "createcontainerconfigerror_bad_configmap_key": ["configmap", "key", "createcontainerconfigerror"],
    "crashloop_bad_args": ["crashloopbackoff", "bad argument", "invalid argument", "flag"],
    "oomkilled_limit_too_low": ["oomkilled", "out of memory", "memory limit"],
    "imagepull_bad_tag": ["imagepullbackoff", "errimagepull", "tag", "manifest unknown"],
    "imagepull_registry_auth": ["pull access denied", "unauthorized", "registry", "imagepull"],
    "readiness_probe_failure": ["readiness probe", "not ready"],
    "liveness_probe_failure": ["liveness probe", "probe failed", "restart"],
    "rbac_forbidden": ["forbidden", "rbac", "permission", "serviceaccount"],
    "service_connection_refused": ["connection refused", "service", "port"],
    "pvc_not_found_mountfail": ["persistentvolumeclaim", "not found", "mount"],
    "pvc_pending_missing_storageclass": ["storageclass", "pending", "persistentvolumeclaim"],
    "failedscheduling_insufficient_cpu": ["failedscheduling", "insufficient cpu"],
    "failedscheduling_insufficient_memory": ["failedscheduling", "insufficient memory"],
    "failedscheduling_taint": ["taint", "toleration", "failedscheduling"],
    "nodeselector_mismatch": ["nodeselector", "didn t match", "node selector"],
    "quota_exceeded_pods": ["quota", "exceeded", "pods"],
}

from pathlib import Path

ADAPTER_PATH = Path("agents/models/trained/model_qwen_solver_fast/lora_adapter").resolve()
print("Adapter path:", ADAPTER_PATH)
print("Exists:", ADAPTER_PATH.exists())
print("Files:", list(ADAPTER_PATH.iterdir())[:10])

Adapter path: /home/akash/repos/Multi-Agent-Collaboration-System/akstuff/agents/models/trained/model_qwen_solver_fast/lora_adapter
Exists: True
Files: [PosixPath('/home/akash/repos/Multi-Agent-Collaboration-System/akstuff/agents/models/trained/model_qwen_solver_fast/lora_adapter/README.md'), PosixPath('/home/akash/repos/Multi-Agent-Collaboration-System/akstuff/agents/models/trained/model_qwen_solver_fast/lora_adapter/tokenizer_config.json'), PosixPath('/home/akash/repos/Multi-Agent-Collaboration-System/akstuff/agents/models/trained/model_qwen_solver_fast/lora_adapter/adapter_model.safetensors'), PosixPath('/home/akash/repos/Multi-Agent-Collaboration-System/akstuff/agents/models/trained/model_qwen_solver_fast/lora_adapter/chat_template.jinja'), PosixPath('/home/akash/repos/Multi-Agent-Collaboration-System/akstuff/agents/models/trained/model_qwen_solver_fast/lora_adapter/tokenizer.json'), PosixPath('/home/akash/repos/Multi-Agent-Collaboration-System/akstuff/agents/models/trained/model_qw

In [3]:
def normalize_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9_\-/\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def build_evidence_prompt(row):
    return str(row.get("evidence_text", "")).strip()


def build_target_response(row):
    sections = []

    diagnosis = str(row.get("diagnosis_text", "")).strip()
    fix_plan = str(row.get("fix_plan_text", "")).strip()
    actions = str(row.get("actions_text", "")).strip()
    verification = str(row.get("verification_text", "")).strip()
    rollback = str(row.get("rollback_text", "")).strip()

    if diagnosis:
        sections.append(f"## Diagnosis\n{diagnosis}")
    if fix_plan:
        sections.append(f"## Fix Plan\n{fix_plan}")
    if actions:
        sections.append(f"## Actions\n{actions}")
    if verification:
        sections.append(f"## Verification\n{verification}")
    if rollback:
        sections.append(f"## Rollback\n{rollback}")

    return "\n\n".join(sections)


def build_inference_prompt(row):
    evidence = build_evidence_prompt(row)
    return (
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\n"
        f"Analyze this Kubernetes incident and provide diagnosis, remediation, verification, and rollback guidance.\n\n"
        f"Incident Evidence:\n{evidence}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )


def section_presence_score(pred_text):
    pred_norm = normalize_text(pred_text)
    found = sum(1 for section in SECTION_NAMES if section in pred_norm)
    return found / len(SECTION_NAMES)


def keyword_recall(pred_text, gt_text, min_len=5):
    pred_words = set(word for word in normalize_text(pred_text).split() if len(word) >= min_len)
    gt_words = set(word for word in normalize_text(gt_text).split() if len(word) >= min_len)

    if not gt_words:
        return 1.0 if not pred_words else 0.0

    return len(pred_words & gt_words) / len(gt_words)


def command_recall(pred_text, gt_text):
    pred_cmds = set(
        line.strip() for line in str(pred_text).splitlines()
        if ("kubectl" in line) or ("helm" in line) or ("kustomize" in line)
    )
    gt_cmds = set(
        line.strip() for line in str(gt_text).splitlines()
        if ("kubectl" in line) or ("helm" in line) or ("kustomize" in line)
    )

    if not gt_cmds:
        return 1.0 if not pred_cmds else 0.0

    return len(pred_cmds & gt_cmds) / len(gt_cmds)


def predict_scenario_from_text(text):
    norm = normalize_text(text)
    best_scenario = None
    best_score = -1

    for scenario, hints in SCENARIO_HINTS.items():
        score = sum(1 for hint in hints if hint in norm)
        if score > best_score:
            best_score = score
            best_scenario = scenario

    return best_scenario


def generate_response(model, tokenizer, row, max_new_tokens=384):
    prompt = build_inference_prompt(row)

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SEQ_LEN,
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    pred = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    )

    if "<|im_end|>" in pred:
        pred = pred.split("<|im_end|>")[0]

    return pred.strip()

In [4]:
print("Loading tokenizer and model...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.bfloat16,
    trust_remote_code=True,
)

model = PeftModel.from_pretrained(base_model, str(ADAPTER_PATH))
model.eval()

print("Model loaded.")

Loading tokenizer and model...


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Model loaded.


In [5]:
print("Loading dataset...")

df = pd.read_parquet(DATA_PATH)
df = df[df["scenario_id"] != "dns_resolution_failure"].copy()
df = df.reset_index(drop=True)

df["evidence_text"] = df["evidence_text"].fillna("").astype(str)
for col in TARGET_COLS:
    df[col] = df[col].fillna("").astype(str)

df = df[df["evidence_text"].str.strip() != ""].copy()
df = df[df[TARGET_COLS].apply(lambda x: x.str.strip()).ne("").any(axis=1)].copy()
df = df.reset_index(drop=True)

dataset = Dataset.from_pandas(df, preserve_index=False)
split_ds = dataset.train_test_split(test_size=TEST_SIZE, seed=SPLIT_SEED)
test_df = split_ds["test"].to_pandas().reset_index(drop=True)

if N_EVAL is not None:
    test_df = test_df.sample(n=min(N_EVAL, len(test_df)), random_state=SPLIT_SEED).reset_index(drop=True)

print(f"Test rows: {len(test_df):,}")
print(test_df["scenario_id"].value_counts().sort_index().to_string())

Loading dataset...
Test rows: 1,700
scenario_id
crashloop_bad_args                              102
createcontainerconfigerror_bad_configmap_key     97
createcontainerconfigerror_missing_secret       110
failedscheduling_insufficient_cpu               105
failedscheduling_insufficient_memory             94
failedscheduling_taint                          105
imagepull_bad_tag                                90
imagepull_registry_auth                         100
liveness_probe_failure                          100
nodeselector_mismatch                           110
oomkilled_limit_too_low                         103
pvc_not_found_mountfail                          85
pvc_pending_missing_storageclass                 99
quota_exceeded_pods                              95
rbac_forbidden                                   96
readiness_probe_failure                         103
service_connection_refused                      106


In [ ]:
# print("\nRunning generation on test set...")
# results = []

# for idx, row in test_df.iterrows():
#     pred = generate_response(model, tokenizer, row)
#     gt = build_target_response(row)

#     pred_scenario = predict_scenario_from_text(pred)
#     scenario_match = int(pred_scenario == row["scenario_id"])

#     result = {
#         "row_idx": idx,
#         "scenario_id": row["scenario_id"],
#         "pred_scenario": pred_scenario,
#         "scenario_match": scenario_match,
#         "section_score": section_presence_score(pred),
#         "keyword_recall": keyword_recall(pred, gt),
#         "command_recall": command_recall(pred, gt),
#         "ground_truth": gt,
#         "prediction": pred,
#     }
#     results.append(result)

#     if (idx + 1) % 20 == 0:
#         print(f"Completed {idx + 1}/{len(test_df)}")


# res_df = pd.DataFrame(results)

test_df_small = test_df.sample(n=min(400, len(test_df)), random_state=42).reset_index(drop=True)

print("\nRunning generation on test subset...")
results = []

for idx, row in test_df_small.iterrows():
    pred = generate_response(model, tokenizer, row)
    gt = build_target_response(row)

    pred_scenario = predict_scenario_from_text(pred)
    scenario_match = int(pred_scenario == row["scenario_id"])

    result = {
        "row_idx": idx,
        "scenario_id": row["scenario_id"],
        "pred_scenario": pred_scenario,
        "scenario_match": scenario_match,
        "section_score": section_presence_score(pred),
        "keyword_recall": keyword_recall(pred, gt),
        "command_recall": command_recall(pred, gt),
        "ground_truth": gt,
        "prediction": pred,
    }
    results.append(result)

    print(f"Completed {idx + 1}/{len(test_df_small)}")

res_df = pd.DataFrame(results)


Running generation on test subset...
Completed 1/20
Completed 2/20
Completed 3/20
Completed 4/20
Completed 5/20
Completed 6/20
Completed 7/20
Completed 8/20
Completed 9/20
Completed 10/20
Completed 11/20
Completed 12/20
Completed 13/20
Completed 14/20
Completed 15/20
Completed 16/20
Completed 17/20
Completed 18/20
Completed 19/20
Completed 20/20


In [8]:
print("\nComputing BERTScore...")
predictions = res_df["prediction"].tolist()
references = res_df["ground_truth"].tolist()

P, R, F1 = bertscore_score(
    predictions,
    references,
    lang="en",
    verbose=True,
)

res_df["bertscore_p"] = P.cpu().numpy()
res_df["bertscore_r"] = R.cpu().numpy()
res_df["bertscore_f1"] = F1.cpu().numpy()


# =========================================================
# ROUGE
# =========================================================
print("Computing ROUGE...")
rouge = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

rouge1_f = []
rouge2_f = []
rougeL_f = []

for pred, ref in zip(predictions, references):
    scores = rouge.score(ref, pred)
    rouge1_f.append(scores["rouge1"].fmeasure)
    rouge2_f.append(scores["rouge2"].fmeasure)
    rougeL_f.append(scores["rougeL"].fmeasure)

res_df["rouge1_f"] = rouge1_f
res_df["rouge2_f"] = rouge2_f
res_df["rougeL_f"] = rougeL_f


Computing BERTScore...


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

calculating scores...
computing bert embedding.


  0%|          | 0/1 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 0.58 seconds, 34.41 sentences/sec
Computing ROUGE...


In [9]:
print("Computing BLEU...")
bleu_scores = []
for pred, ref in zip(predictions, references):
    bleu = sacrebleu.sentence_bleu(pred, [ref])
    bleu_scores.append(bleu.score / 100.0)

res_df["bleu"] = bleu_scores

Computing BLEU...


In [10]:
summary = {
    "n_test_examples": int(len(res_df)),
    "scenario_match_accuracy": float(res_df["scenario_match"].mean()),
    "section_score_mean": float(res_df["section_score"].mean()),
    "keyword_recall_mean": float(res_df["keyword_recall"].mean()),
    "command_recall_mean": float(res_df["command_recall"].mean()),
    "bertscore_p_mean": float(res_df["bertscore_p"].mean()),
    "bertscore_r_mean": float(res_df["bertscore_r"].mean()),
    "bertscore_f1_mean": float(res_df["bertscore_f1"].mean()),
    "rouge1_f_mean": float(res_df["rouge1_f"].mean()),
    "rouge2_f_mean": float(res_df["rouge2_f"].mean()),
    "rougeL_f_mean": float(res_df["rougeL_f"].mean()),
    "bleu_mean": float(res_df["bleu"].mean()),
}

per_scenario_accuracy = (
    res_df.groupby("scenario_id")["scenario_match"]
    .mean()
    .sort_values()
    .to_dict()
)

print("\n================ SUMMARY ================")
for k, v in summary.items():
    print(f"{k}: {v}")

print("\n=========== PER-SCENARIO ACCURACY ==========")
for k, v in per_scenario_accuracy.items():
    print(f"{k}: {v:.4f}")



================ SUMMARY ================
n_test_examples: 20
scenario_match_accuracy: 0.6
section_score_mean: 0.97
keyword_recall_mean: 0.4273265567830785
command_recall_mean: 0.0
bertscore_p_mean: 0.8170992732048035
bertscore_r_mean: 0.864767849445343
bertscore_f1_mean: 0.839976966381073
rouge1_f_mean: 0.4275165278304952
rouge2_f_mean: 0.1729146292693306
rougeL_f_mean: 0.2834348169275429
bleu_mean: 0.11169754633455312

=========== PER-SCENARIO ACCURACY ==========
failedscheduling_insufficient_cpu: 0.0000
imagepull_bad_tag: 0.0000
nodeselector_mismatch: 0.0000
readiness_probe_failure: 0.0000
imagepull_registry_auth: 0.3333
crashloop_bad_args: 0.5000
liveness_probe_failure: 1.0000
failedscheduling_insufficient_memory: 1.0000
oomkilled_limit_too_low: 1.0000
pvc_not_found_mountfail: 1.0000
pvc_pending_missing_storageclass: 1.0000
quota_exceeded_pods: 1.0000


In [11]:
OUT_DIR.mkdir(parents=True, exist_ok=True)

res_df.to_csv(OUT_DIR / "per_example_results.csv", index=False)

with open(OUT_DIR / "summary_metrics.json", "w") as f:
    json.dump(
        {
            "summary": summary,
            "per_scenario_accuracy": per_scenario_accuracy,
        },
        f,
        indent=2,
    )

print(f"\nSaved results to: {OUT_DIR}")


Saved results to: agents/models/trained/model_qwen_solver/eval
